# BÀI TẬP PHẦN 1: CÁC THUẬT TOÁN TÌM KIẾM TRÊN ĐỒ THỊ

**Họ tên:** Nguyễn Ngọc Hải

**MSSV:** 24880218

## Hướng dẫn
Phần này yêu cầu bạn cài đặt các thuật toán tìm kiếm cơ bản (BFS, DFS, Greedy BFS, UCS, A*) trên cấu trúc dữ liệu đồ thị dạng danh sách kề.

**Yêu cầu:**
1. **Phiên bản Tiêu chuẩn:** Chỉ sử dụng `list` cơ bản của Python. Không dùng `heapq`, `deque`. Với Priority Queue, phải duyệt danh sách để tìm phần tử nhỏ nhất.
2. **Phiên bản Tối ưu (Optimized):** Sử dụng các cấu trúc dữ liệu hiệu quả (`collections.deque` cho Queue, `heapq` cho Priority Queue).

**Lưu ý quan trọng:**
- Bạn cần có file `graph.txt` nằm cùng thư mục với notebook này để chạy code.
- Các cell kiểm tra (Judge Cells) và cell Benchmarking sẽ **đọc trực tiếp dữ liệu từ file `graph.txt`** bằng hàm `read_graph_input` của bạn.
- **Tie-breaking:** Nếu giá trị ưu tiên bằng nhau, hãy chọn đỉnh có **index nhỏ hơn**.
- Đảm bảo hàm tìm kiếm trả về `None` nếu không tìm thấy đường đi.

In [ ]:
import sys
import time
import heapq
import os
from collections import deque
import matplotlib.pyplot as plt 
%matplotlib inline

### 1.0. Hàm đọc input

Cài đặt hàm đọc đồ thị từ file.
**Format file:**
- Dòng 1: `n m` (số đỉnh, số cạnh)
- Dòng 2: `h0 h1 ... hn-1` (heuristics)
- m dòng sau: `u v w` (cạnh u->v trọng số w)

**Ví dụ:**

```text
4 5
7 2 1 0
0 1 1
0 2 4
1 2 2
1 3 5
2 3 1
```

In [ ]:
def read_graph_input(file_path):
    """
    Đọc đồ thị từ file_path.
    Trả về: (graph, heuristics, start, goal)
    graph: Dictionary {u: [(v, w), ...]}
    heuristics: List [h0, h1, ...]
    start: 0
    goal: n-1
    """
    # YOUR CODE HERE
    # Mở file
    with open(file_path, "r", encoding="utf-8") as f:
        # Đọc dòng 1 trong file để lấy giá trị n và m
        n, m = map(int, (f.readline().split()))

        # Set giá trị cho start và goal
        start, goal = 0, n - 1

        # Khởi tạo dictionary graph có các value là empty list cho các đỉnh trong graph
        graph = {u: [] for u in range(n)}

        # Đọc dòng 2 trong file để lấy list heuristic
        heuristics = list(map(int, f.readline().split()))

        # Loop qua toàn bộ các dòng còn lại trong file
        for line in f:
            # Tách line đọc được thành list các word
            temp = line.split()
            # Kiểm tra nếu line là 1 dòng trống
            if not temp:
                continue
            # Đọc line và tách lấy 3 giá trị u, v, w
            u, v, w = map(int, temp)
            # Append vào list value của đỉnh key tương ứng
            graph[u].append((v, w))

    # Return kết quả
    return graph, heuristics, start, goal

## 1.a. Cài đặt thuật toán Tiêu chuẩn

Trong phần này, bạn **KHÔNG ĐƯỢC** sử dụng `collections.deque` hay `heapq`. Bạn chỉ được dùng `list` của Python.
- **Queue (BFS):** Dùng `list`, lấy phần tử đầu bằng `.pop(0)`.
- **Priority Queue (Greedy, UCS, A*):** Dùng `list`, mỗi lần cần lấy phần tử nhỏ nhất thì duyệt toàn bộ list để tìm (Linear Scan).
- **Tie-break:** Nếu giá trị so sánh bằng nhau, ưu tiên đỉnh có index nhỏ hơn.

In [ ]:
def bfs(graph, start, goal):
    """BFS sử dụng list làm queue (pop(0))."""
    # YOUR CODE HERE
    # Kiểm tra nếu start cũng là goal
    if start == goal:
        return [start]

    # Dict trace dùng để search ngược đường đi
    trace = dict()
    # Add Start vào queue
    queue = [start]
    # Sử dụng set để tăng tốc độ search so với list
    # Set visited dùng để đánh dấu các đỉnh đã nhìn thấy (bao gồm đã duyệt)
    visited = {start}

    # Loop đến khi queue rỗng
    while queue:        
        # Lấy đỉnh đầu queue ra khỏi queue (FIFO)
        u = queue.pop(0)

        # Loop qua từng cặp đỉnh kề và trọng số
        # Thực hiện sorted graph để đảm bảo index tăng dần
        for v, _ in sorted(graph[u]):
            # Kiểm tra nếu v đã xuất hiện rồi thì bỏ qua
            if v in visited:
                continue
            # Thêm đỉnh v vào queue và visited
            queue.append(v)
            visited.add(v)
            # Tạo một key, value trong trace dict để đánh dấu đỉnh v do đỉnh u đi tới
            trace[v] = u

            # Dừng nếu nhìn thấy goal
            if v == goal:
                # Tìm thấy đường từ start đến goal, tạo path và trả về
                path = [goal]

                # Loop đi ngược từ goal đến start
                while path[-1] != start:
                    # Thêm đỉnh vào liền trước vào path
                    path.append(trace[path[-1]])

                # Reverse path và return
                return path[::-1]

    # Không tìm thấy path trả về None
    return None

def dfs(graph, start, goal):
    """DFS sử dụng list làm stack."""
    # YOUR CODE HERE
    # Kiểm tra nếu start cũng là goal
    if start == goal:
        return [start]

    # Khởi tạo
    trace = {}       # Dict dùng để trace back tạo path
    visited = set()  # Set đánh dấu các node đã visit 
    stack = [start]  # stack để duyệt DFS

    # Loop đến khi stack empty
    while stack:
        # Lấy node ở đỉnh stack
        u = stack.pop()

        # Node đã visit rồi -> bỏ qua, vì chỉ có 2 trường hợp:
        # TH1: Node không dẫn đến goal -> visit lại cũng không đến goal
        # TH2: Node có thể dẫn đến goal, nhưng cần phải đi qua node con khác 
        #      -> chỉ cần thăm các node con chưa xuất hiện tron visited thôi
        if u in visited:
            continue

        # Đánh dấu node đã visit
        visited.add(u)

        # Nếu là goal thì dừng và trả về path
        if u == goal:
            path = [u]
            while path[-1] != start:
                path.append(trace[path[-1]])
            return path[::-1]

        # Từ node đang duyệt, add các node kề vào stack với điều kiện node index nhỏ nằm ở đỉnh stack
        for v, _ in sorted(graph[u], reverse=True):
            if v not in visited:
                stack.append(v)
                # Tạo cặp node: node cha, trong dict
                trace[v] = u

    # Không tim thấy path -> Return None
    return None

def greedy_best_first_search(graph, heuristics, start, goal):
    """Greedy Best-First Search tự tìm min trong list."""
    # YOUR CODE HERE
    raise NotImplementedError()

def ucs(graph, start, goal):
    """Uniform Cost Search tự tìm min cost trong list."""
    # YOUR CODE HERE
    raise NotImplementedError()

def a_star(graph, heuristics, start, goal):
    """A* tự tìm min f trong list."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [8]:
# TEST CELL: BFS Standard (5 points)


In [ ]:
# TEST CELL: DFS Standard (5 points)


In [ ]:
# TEST CELL: Greedy Standard (5 points)


In [ ]:
# TEST CELL: UCS Standard (5 points)


In [ ]:
# TEST CELL: A* Standard (5 points)


### 1.2. Cài đặt thuật toán Tối ưu (Optimized)

Trong phần này, hãy cải thiện hiệu năng bằng cách sử dụng:
- `collections.deque` cho BFS.
- `heapq` cho Greedy, UCS, và A*.

Tên hàm sẽ có hậu tố `_optimized`.

In [ ]:
def bfs_optimized(graph, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError()

def dfs_optimized(graph, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError()

def greedy_optimized(graph, heuristics, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError()

def ucs_optimized(graph, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError()

def astar_optimized(graph, heuristics, start, goal):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# TEST CELL: BFS Optimized (5 points)


In [ ]:
# TEST CELL: DFS Optimized (5 points)


In [ ]:
# TEST CELL: Greedy Optimized (5 points)


In [ ]:
# TEST CELL: UCS Optimized (5 points)


In [ ]:
# TEST CELL: A* Optimized (5 points)


### 1.3. So sánh hiệu năng (Benchmarking)

Code dưới đây sẽ so sánh thời gian chạy giữa phiên bản **Tiêu chuẩn** và **Tối ưu** trên dữ liệu đọc từ `./graph.txt`.

In [15]:
def benchmark_algorithms():
    try:
        graph, heuristics, start, goal = read_graph_input('./graph.txt')
    except FileNotFoundError:
        print("Lỗi: Không tìm thấy file './graph.txt'. Vui lòng tạo file này để chạy benchmark.")
        return

    # (Algorithm Name, Std Func, Optimized Func, Arguments)
    algos = [
        ("BFS", bfs, bfs_optimized, (graph, start, goal)),
        ("DFS", dfs, dfs_optimized, (graph, start, goal)),
        ("Greedy", greedy_best_first_search, greedy_optimized, (graph, heuristics, start, goal)),
        ("UCS", ucs, ucs_optimized, (graph, start, goal)),
        ("A*", a_star, astar_optimized, (graph, heuristics, start, goal))
    ]
    
    labels = []
    t_std = []
    t_opt = []
    
    print(f"{'Algorithm':<10} | {'Standard (s)':<12} | {'Optimized (s)':<12} | {'Improvement':<12}")
    print("-"*55)

    for name, s_func, o_func, args in algos:
        try:
            # Standard
            st = time.time()
            s_func(*args)
            ts = time.time() - st
            
            # Optimized
            st = time.time()
            o_func(*args)
            to = time.time() - st
            
            labels.append(name)
            t_std.append(ts)
            t_opt.append(to)
            
            improv = ts / to if to > 0 else 0
            print(f"{name:<10} | {ts:.6f}     | {to:.6f}     | {improv:.2f}x")
            
        except Exception as e:
            print(f"{name:<10} | Error: {e}")

    if labels:
        x = range(len(labels))
        width = 0.35
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.bar([i - width/2 for i in x], t_std, width, label='Standard', color='salmon')
        ax.bar([i + width/2 for i in x], t_opt, width, label='Optimized', color='skyblue')
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel('Time (seconds)')
        ax.set_title('Performance Comparison')
        ax.legend()
        plt.show()

benchmark_algorithms()

NameError: name 'bfs_optimized' is not defined

### Câu hỏi lý thuyết (10 điểm)
So sánh ưu nhược điểm của BFS và DFS trong trường hợp đồ thị vô hạn hoặc rất lớn.

Câu trả lời:

YOUR ANSWER HERE